# PyTorch Fundamentals and Training Loop

Practice tensor construction and reshaping, NumPy interchange, autograd, Dataset/DataLoader, nn.Module, correct train/eval modes, and loss aggregation.

- **Study time:** 45-60 minutes
- **Prerequisites:** NumPy shapes, derivatives, and Python classes
- **Mode:** `optional`
- **Data policy:** no downloads; seeded synthetic tensors only; any checkpoint path resolves outside the vault
- **Provenance:** consolidated from the legacy PyTorch introduction and training-loop drills

Output convention: every retained textual result begins with a label that identifies the operation that produced it.


In [ ]:
import sys
from pathlib import Path


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the DataCoding project")


PROJECT_ROOT = find_project_root()
source_dir = str(PROJECT_ROOT / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

In [ ]:
import random

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from datacoding.config import external_path


def seed_all(seed=51):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def show(label, value):
    print(f"\n--- {label} ---\n{value}")


seed_all()
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
show("Environment | torch version", torch.__version__)
show("Environment | selected device", device)

## 1. Tensor construction, shape operations, and combining

`torch.cat` joins an existing dimension; `torch.stack` creates a new one. `from_numpy` shares CPU memory with its NumPy input, while `torch.tensor` copies. Before converting a model result to NumPy, use `detach().cpu().numpy()`.


In [ ]:
X = torch.arange(12, dtype=torch.float32).reshape(3, 4)
bias = torch.tensor([1.0, 2.0, 3.0, 4.0])
shifted = X + bias
expanded = X.unsqueeze(0)  # (1, 3, 4)
permuted = expanded.permute(0, 2, 1)  # (1, 4, 3)
concatenated = torch.cat([X, X], dim=0)  # (6, 4)
stacked = torch.stack([X, X], dim=0)  # (2, 3, 4)

numpy_source = np.arange(6, dtype=np.float32).reshape(2, 3)
shared_tensor = torch.from_numpy(numpy_source)
copied_tensor = torch.tensor(numpy_source)
numpy_source[0, 0] = -1.0
detached_numpy = shifted.detach().cpu().numpy()

show("Tensor | X shape/dtype/device", (tuple(X.shape), X.dtype, X.device))
show("Broadcast | X + bias shape", tuple(shifted.shape))
show("Broadcast | shifted values", shifted)
show("Shape | unsqueeze then permute", (tuple(expanded.shape), tuple(permuted.shape)))
show(
    "Combine | cat existing dim versus stack new dim",
    (tuple(concatenated.shape), tuple(stacked.shape)),
)
show(
    "NumPy boundary | from_numpy shares, tensor copies",
    (shared_tensor[0, 0].item(), copied_tensor[0, 0].item()),
)
show("NumPy boundary | detached CPU array shape", detached_numpy.shape)

## 2. Autograd and gradient accumulation


In [ ]:
weight = torch.tensor(2.0, requires_grad=True)
loss = (weight * 3.0 - 7.0) ** 2
loss.backward()
first_gradient = weight.grad.item()
second_loss = (weight * 3.0 - 7.0) ** 2
second_loss.backward()
accumulated_gradient = weight.grad.item()
weight.grad.zero_()

show("Autograd | scalar loss", loss.item())
show("Autograd | first gradient", first_gradient)
show("Autograd | accumulated after second backward", accumulated_gradient)
show("Autograd | gradient after zero", weight.grad.item())

## 3. Dataset and DataLoader contract


In [ ]:
n_rows, n_features = 1_200, 5
features = torch.randn(n_rows, n_features)
true_weight = torch.tensor([1.5, -2.0, 0.5, 3.0, -1.0]).reshape(-1, 1)
targets = features @ true_weight + 0.4 + 0.2 * torch.randn(n_rows, 1)

train_dataset = TensorDataset(features[:900], targets[:900])
validation_dataset = TensorDataset(features[900:], targets[900:])
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=128, shuffle=False)
sample_X, sample_y = next(iter(train_loader))

show("DataLoader | feature and target batch shapes", (tuple(sample_X.shape), tuple(sample_y.shape)))

## 4. Model and canonical loops


In [ ]:
class TinyRegressor(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, X):
        return self.network(X)


def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0.0
    total_examples = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad(set_to_none=True)
        prediction = model(X_batch)
        loss = loss_fn(prediction, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X_batch)
        total_examples += len(X_batch)
    return total_loss / total_examples


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            loss = loss_fn(model(X_batch), y_batch)
            total_loss += loss.item() * len(X_batch)
            total_examples += len(X_batch)
    return total_loss / total_examples

In [ ]:
model = TinyRegressor(n_features).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
loss_fn = nn.MSELoss()

history = []
for epoch in range(1, 7):
    train_loss = train_epoch(model, train_loader, optimizer, loss_fn)
    validation_loss = evaluate(model, validation_loader, loss_fn)
    history.append((train_loss, validation_loss))
    print(
        f"Training | epoch={epoch:02d} train_mse={train_loss:.4f} validation_mse={validation_loss:.4f}"
    )

show("Training | first and final loss pairs", (history[0], history[-1]))

## 5. Save learned state outside the vault


In [ ]:
checkpoint_path = external_path("models", "tiny_regressor_state.pt", create_parent=True)
torch.save(model.state_dict(), checkpoint_path)

restored = TinyRegressor(n_features).to(device)
restored.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
restored_loss = evaluate(restored, validation_loader, loss_fn)

assert concatenated.shape == (6, 4)
assert stacked.shape == (2, 3, 4)
assert accumulated_gradient == 2 * first_gradient
show("Checkpoint | external path", checkpoint_path)
show("Checkpoint | restored validation MSE", restored_loss)
show(
    "Training checks | status", "model modes, no-grad evaluation, and external state_dict verified"
)